In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!git clone https://github.com/RGenDiff/hgvt.git

Cloning into 'hgvt'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 37 (delta 6), reused 37 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 872.07 KiB | 5.74 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [2]:
%cd hgvt

/kaggle/working/hgvt


In [3]:
!pip install -e .

Obtaining file:///kaggle/working/hgvt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hgvt (pyproject.toml) ... done
  Created wheel for hgvt: filename=hgvt-0.0.1-0.editable-py3-none-any.whl size=8123 sha256=a9c88342616493263b15af4c64123459ef9da9e64296549502b4a3277ca50f24
  Stored in directory: /tmp/pip-ephem-wheel-cache-8r5i9v_7/wheels/6b/51/0e/451b5fca449bd04de3cf447b7197a9f9c27db46ad93ab7ba9e
Successfully built hgvt


In [4]:
import torch
print(torch.__version__, torch.cuda.is_available())

2.10.0+cu128 True


In [5]:
# unzip your upload (or use the git clone you already did)
!unzip -q hgvt-main.zip && cd hgvt-main

# core package
!pip install -e .

# additional deps actually imported by the code but NOT in pyproject.toml:
!pip install pytorch-lightning scipy kornia tqdm packaging

# only needed if you want the ImageNet/WebDataset path (not CIFAR):
!pip install webdataset

unzip:  cannot find or open hgvt-main.zip, hgvt-main.zip.zip or hgvt-main.zip.ZIP.
Obtaining file:///kaggle/working/hgvt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hgvt (pyproject.toml) ... done
  Created wheel for hgvt: filename=hgvt-0.0.1-0.editable-py3-none-any.whl size=8123 sha256=cc82d6484c859f7b612704f8dc502ad86326417aaedc0ae56bc6ef0e00454c58
  Stored in directory: /tmp/pip-ephem-wheel-cache-44occypi/wheels/6b/51/0e/451b5fca449bd04de3cf447b7197a9f9c27db46ad93ab7ba9e
Successfully built hgvt
  Attempting uninstall: hgvt
    Found existing installation: hgvt 0.0.1
    Uninstalling hgvt-0.0.1:
      Successfully uninstalled hgvt-0.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.0 MB/s eta 0:00:00


In [6]:
!pip install -e .
!pip install pytorch-lightning scipy kornia tqdm packaging webdataset

Obtaining file:///kaggle/working/hgvt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hgvt (pyproject.toml) ... done
  Created wheel for hgvt: filename=hgvt-0.0.1-0.editable-py3-none-any.whl size=8123 sha256=7f32ca7017470d3f1c1171ced23d18f02346ae5d3b9678290477f309d88cd758
  Stored in directory: /tmp/pip-ephem-wheel-cache-zoby4aj1/wheels/6b/51/0e/451b5fca449bd04de3cf447b7197a9f9c27db46ad93ab7ba9e
Successfully built hgvt
  Attempting uninstall: hgvt
    Found existing installation: hgvt 0.0.1
    Uninstalling hgvt-0.0.1:
      Successfully uninstalled hgvt-0.0.1


In [7]:
path = "/kaggle/working/hgvt/hgvt/hypergraph.py"
with open(path) as f:
    content = f.read()
content = content.replace(
    "from pytorch_lightning.utilities.distributed import rank_zero_only",
    "from pytorch_lightning.utilities.rank_zero import rank_zero_only"
)
with open(path, "w") as f:
    f.write(content)
print("patched 1/5")

patched 1/5


In [8]:
path = "/kaggle/working/hgvt/hgvt/hypergraph.py"
with open(path) as f:
    content = f.read()

old = """    def stop_validation(self, reduce_time=False):

        # get the world size
        world_size = dist.get_world_size()
        
        # Concatenate features from all batches
        losses_m = torch.tensor([self.losses_m.avg], device=self.device)
        top1_m = torch.tensor([self.top1_m.avg], device=self.device)
        top5_m = torch.tensor([self.top5_m.avg], device=self.device)
        kentropy_m = torch.tensor([self.k_entropy_m.avg], device=self.device)
        kdiversity_m = torch.tensor([self.k_diversity_m.avg], device=self.device)
            
        # allocate the gather features from all GPUs
        gathered_losses_m = [torch.zeros_like(losses_m) for _ in range(world_size)]
        gathered_top1_m = [torch.zeros_like(top1_m) for _ in range(world_size)]
        gathered_top5_m = [torch.zeros_like(top5_m) for _ in range(world_size)]
        gathered_kentropy_m = [torch.zeros_like(kentropy_m) for _ in range(world_size)]
        gathered_kdiversity_m = [torch.zeros_like(kdiversity_m) for _ in range(world_size)]
        
        # perform the gather
        dist.all_gather(gathered_losses_m, losses_m)
        dist.all_gather(gathered_top1_m, top1_m)
        dist.all_gather(gathered_top5_m, top5_m)
        dist.all_gather(gathered_kentropy_m, kentropy_m)
        dist.all_gather(gathered_kdiversity_m, kdiversity_m)
        
        # Concatenate features from all GPUs and mean
        final_losses_m = torch.cat(gathered_losses_m, dim=0).mean()
        final_top1_m = torch.cat(gathered_top1_m, dim=0).mean()
        final_top5_m = torch.cat(gathered_top5_m, dim=0).mean()
        final_kentropy_m = torch.cat(gathered_kentropy_m, dim=0).mean()
        final_kdiversity_m = torch.cat(gathered_kdiversity_m, dim=0).mean()

        if reduce_time:
            batch_time_m = torch.tensor([self.batch_time_m.avg], device=self.device)
            gathered_batch_time_m = [torch.zeros_like(batch_time_m) for _ in range(world_size)]
            dist.all_gather(gathered_batch_time_m, batch_time_m)
            final_batch_time_m = torch.cat(gathered_batch_time_m, dim=0).mean()
        else:
            final_batch_time_m = None"""

new = """    def stop_validation(self, reduce_time=False):

        # Original code assumed a DDP process group is always initialized.
        # On a single-GPU/no-strategy run there is no default process group,
        # so dist.get_world_size()/dist.all_gather() would raise. Detect that
        # case and skip the cross-GPU gather entirely (world_size=1: the
        # local averages are already the final answer).
        distributed = dist.is_available() and dist.is_initialized()
        world_size = dist.get_world_size() if distributed else 1

        # Concatenate features from all batches
        losses_m = torch.tensor([self.losses_m.avg], device=self.device)
        top1_m = torch.tensor([self.top1_m.avg], device=self.device)
        top5_m = torch.tensor([self.top5_m.avg], device=self.device)
        kentropy_m = torch.tensor([self.k_entropy_m.avg], device=self.device)
        kdiversity_m = torch.tensor([self.k_diversity_m.avg], device=self.device)

        def gather_or_local(tensor):
            if not distributed:
                return [tensor]
            gathered = [torch.zeros_like(tensor) for _ in range(world_size)]
            dist.all_gather(gathered, tensor)
            return gathered

        gathered_losses_m = gather_or_local(losses_m)
        gathered_top1_m = gather_or_local(top1_m)
        gathered_top5_m = gather_or_local(top5_m)
        gathered_kentropy_m = gather_or_local(kentropy_m)
        gathered_kdiversity_m = gather_or_local(kdiversity_m)

        # Concatenate features from all GPUs and mean
        final_losses_m = torch.cat(gathered_losses_m, dim=0).mean()
        final_top1_m = torch.cat(gathered_top1_m, dim=0).mean()
        final_top5_m = torch.cat(gathered_top5_m, dim=0).mean()
        final_kentropy_m = torch.cat(gathered_kentropy_m, dim=0).mean()
        final_kdiversity_m = torch.cat(gathered_kdiversity_m, dim=0).mean()

        if reduce_time:
            batch_time_m = torch.tensor([self.batch_time_m.avg], device=self.device)
            gathered_batch_time_m = gather_or_local(batch_time_m)
            final_batch_time_m = torch.cat(gathered_batch_time_m, dim=0).mean()
        else:
            final_batch_time_m = None"""

assert old in content, "pattern not found — file may already be patched or differs"
content = content.replace(old, new)
with open(path, "w") as f:
    f.write(content)
print("patched 2/5")

patched 2/5


In [9]:
path = "/kaggle/working/hgvt/hgvt/data/torchvision_data_module.py"
with open(path) as f:
    content = f.read()

old = """        # Apply the transform multiple times
        imgs = []
        for _ in range(self.repeat_aug):
            ximg = self.transform(pil_image)
            if self.use_prefetcher:
                ximg = torch.from_numpy(ximg)
            imgs.append(ximg)"""

new = """        # Apply the transform multiple times
        imgs = []
        for _ in range(self.repeat_aug):
            ximg = self.transform(pil_image)
            if self.use_prefetcher and isinstance(ximg, np.ndarray):
                ximg = torch.from_numpy(ximg)
            imgs.append(ximg)"""

assert old in content, "pattern not found — check for whitespace differences"
content = content.replace(old, new)
with open(path, "w") as f:
    f.write(content)
print("patched 3/5")

patched 3/5


In [10]:
path = "/kaggle/working/hgvt/hgvt/data/prefetch_loader.py"
with open(path) as f:
    content = f.read()

old = """    # don't include the len property since wds doesn't have it
    #def __len__(self):
    #    return len(self.loader)"""

new = """    # don't include the len property since wds doesn't have it
    #def __len__(self):
    #    return len(self.loader)

    @property
    def batch_sampler(self):
        return getattr(self.loader, "batch_sampler", None)

    @property
    def sampler(self):
        return getattr(self.loader, "sampler", None)"""

assert old in content, "pattern not found — check for whitespace differences"
content = content.replace(old, new)
with open(path, "w") as f:
    f.write(content)
print("patched 4/5")

patched 4/5


In [11]:
path = "/kaggle/working/hgvt/configs/hgvt_mu.yaml"
with open(path) as f:
    content = f.read()
content = content.replace("opt: fusedadamw", "opt: adamw")
with open(path, "w") as f:
    f.write(content)
print("patched 5/5")

patched 5/5


In [12]:
%%writefile /kaggle/working/hgvt/train.py
"""
train.py — CLI entry point for HgVT training.

This file is NOT part of the original RGenDiff/hgvt release (the repo's
README references it, but it isn't included in the repo as published).
It wires together the existing hgvt.hypergraph.HyperGraph LightningModule
and the data modules in hgvt/data/ into a runnable PyTorch Lightning
training script, following the CLI shown in the README:

    PRECISION="bf16" && \
    export ATTN_PRECISION="fp32" && \
    export USE_XFORMERS=1 && \
    export USE_APEX=0 && \
    python train.py \
        --precision "$PRECISION" \
        --gpus 2 \
        --name name_of_run \
        --logdir path/to/log/dir \
        --config config.yaml

Notes on changes vs. the README command:
  - `--gpus` is accepted for compatibility with the README, but is
    translated internally to accelerator="gpu"/devices=N, since the
    `gpus=` Trainer argument was removed in PyTorch Lightning 2.0.
  - `--gpus 0` (or no GPU available) falls back to CPU training.
  - On Kaggle/notebook environments, multi-GPU DDP can be unstable, so
    devices>1 uses strategy="ddp" with find_unused_parameters=False;
    for a single notebook GPU just use --gpus 1 (the common case).
"""

import argparse
import os
import time

from omegaconf import OmegaConf
import torch
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor

from hgvt.util import instantiate_from_config


def get_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--config", type=str, required=True,
        help="Path to a config yaml (e.g. configs/hgvt_mu.yaml).",
    )
    parser.add_argument(
        "--name", type=str, default=None,
        help="Name for this run. Used as the subdirectory under --logdir.",
    )
    parser.add_argument(
        "--logdir", type=str, default="logs",
        help="Root directory for logs and checkpoints.",
    )
    parser.add_argument(
        "--gpus", type=int, default=1,
        help="Number of GPUs to use. 0 = CPU.",
    )
    parser.add_argument(
        "--precision", type=str, default="32",
        help="Training precision: 32, 16, or bf16.",
    )
    parser.add_argument(
        "--resume", type=str, default=None,
        help="Path to a checkpoint (.ckpt) to resume training from.",
    )
    parser.add_argument(
        "--seed", type=int, default=21,
        help="Random seed.",
    )
    parser.add_argument(
        "--max_steps", type=int, default=None,
        help="Override the config's max training steps (useful for smoke tests).",
    )
    parser.add_argument(
        "--limit_train_batches", type=str, default=None,
        help="Limit train batches per epoch. Pass an int (e.g. 4) or a float "
             "fraction (e.g. 0.01). Useful for a quick smoke test on a small "
             "slice of data without touching the dataset/config.",
    )
    parser.add_argument(
        "--limit_val_batches", type=str, default=None,
        help="Limit val batches per epoch, same format as --limit_train_batches.",
    )
    return parser


def normalize_precision(precision_arg):
    """Map the README's simple precision strings onto whatever this
    installed PyTorch Lightning version expects. Recent PL versions
    prefer '32-true' / '16-mixed' / 'bf16-mixed', but usually still
    accept the older short forms. We try the short form first and
    fall back to the '-mixed' spelling if construction fails later."""
    mapping = {
        "32": "32",
        "16": "16",
        "bf16": "bf16",
    }
    return mapping.get(precision_arg, precision_arg)


def parse_int_or_float(value):
    if value is None:
        return None
    try:
        return int(value)
    except ValueError:
        return float(value)


def build_trainer(args, trainer_extra_cfg, logger, callbacks):
    use_gpu = args.gpus > 0 and torch.cuda.is_available()
    if args.gpus > 0 and not torch.cuda.is_available():
        print("WARNING: --gpus > 0 requested but no CUDA device is visible. Falling back to CPU.")

    accelerator = "gpu" if use_gpu else "cpu"
    devices = args.gpus if use_gpu else 1

    # strip legacy/incompatible keys that may be present in the yaml's
    # `lightning.trainer` block (written against an older PL version)
    trainer_kwargs = dict(trainer_extra_cfg)
    # strip anything we set explicitly ourselves below, so the yaml's
    # `lightning.trainer` block can't collide with our kwargs
    for key in ("gpus", "strategy", "accelerator", "devices", "precision",
                "max_steps", "limit_train_batches", "limit_val_batches"):
        trainer_kwargs.pop(key, None)

    # smoke-test overrides
    if args.max_steps is not None:
        trainer_kwargs["max_steps"] = args.max_steps
    if args.limit_train_batches is not None:
        trainer_kwargs["limit_train_batches"] = parse_int_or_float(args.limit_train_batches)
    if args.limit_val_batches is not None:
        trainer_kwargs["limit_val_batches"] = parse_int_or_float(args.limit_val_batches)

    if devices > 1:
        # multi-GPU: keep a DDP strategy, but disable unused-parameter
        # detection the same way the original config intended
        trainer_kwargs["strategy"] = "ddp"
    # else: leave strategy unset (defaults to "auto"), which is the
    # only reliable choice for a single-GPU notebook/Kaggle session

    precision = normalize_precision(args.precision)

    try:
        trainer = pl.Trainer(
            accelerator=accelerator,
            devices=devices,
            precision=precision,
            logger=logger,
            callbacks=callbacks,
            **trainer_kwargs,
        )
    except Exception as e:
        # retry once with the '-mixed' precision spelling used by newer PL
        if precision in ("16", "bf16"):
            print(f"Trainer construction failed with precision='{precision}' ({e}); "
                  f"retrying with '{precision}-mixed'.")
            trainer = pl.Trainer(
                accelerator=accelerator,
                devices=devices,
                precision=f"{precision}-mixed",
                logger=logger,
                callbacks=callbacks,
                **trainer_kwargs,
            )
        else:
            raise
    return trainer


def main():
    parser = get_parser()
    args = parser.parse_args()

    pl.seed_everything(args.seed)

    config = OmegaConf.load(args.config)
    model_config = config.model
    data_config = config.data
    lightning_config = config.get("lightning", OmegaConf.create())
    trainer_config = lightning_config.get("trainer", OmegaConf.create())
    ckpt_config = lightning_config.get("modelcheckpoint", OmegaConf.create()).get("params", OmegaConf.create())

    run_name = args.name or os.path.splitext(os.path.basename(args.config))[0]
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_dir = os.path.join(args.logdir, f"{timestamp}_{run_name}")
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    print(f"Run directory: {run_dir}")

    # --- build model ---
    model = instantiate_from_config(model_config)
    if args.resume is not None:
        print(f"Resuming weights from {args.resume}")

    # --- build data module ---
    datamodule = instantiate_from_config(data_config)

    # PrefetchLoaderFromConfig wraps an inner factory data module
    # (e.g. TorchDataModule) in `datamodule.dataset_factory`. PyTorch
    # Lightning only calls `prepare_data()` on the object passed to
    # `trainer.fit()`, so if the inner factory defines its own
    # `prepare_data()` (e.g. to download CIFAR), it is never triggered
    # automatically. Call it explicitly here if present.
    inner_factory = getattr(datamodule, "dataset_factory", None)
    if inner_factory is not None and hasattr(inner_factory, "prepare_data"):
        print("Preparing/downloading dataset via inner data factory...")
        inner_factory.prepare_data()

    # --- logger ---
    logger = TensorBoardLogger(save_dir=args.logdir, name=run_name, version=timestamp)

    # --- callbacks ---
    checkpoint_kwargs = dict(ckpt_config)
    checkpoint_kwargs.setdefault("dirpath", ckpt_dir)
    checkpoint_kwargs.setdefault("filename", "{epoch}-{step}")
    monitor = model_config.params.get("monitor", None)
    if monitor is not None:
        checkpoint_kwargs.setdefault("monitor", monitor)
        checkpoint_kwargs.setdefault("mode", model_config.params.get("mon_mode", "min"))
    checkpoint_kwargs.setdefault("save_top_k", model_config.params.get("save_top_k", 1))
    checkpoint_callback = ModelCheckpoint(**checkpoint_kwargs)

    lr_monitor = LearningRateMonitor(logging_interval="step")

    callbacks = [checkpoint_callback, lr_monitor]

    # --- trainer ---
    trainer = build_trainer(args, trainer_config, logger, callbacks)

    # --- go ---
    trainer.fit(model, datamodule=datamodule, ckpt_path=args.resume)


if __name__ == "__main__":
    main()

Writing /kaggle/working/hgvt/train.py


In [13]:
!find /kaggle/input -iname "*cifar*" -maxdepth 4

find: warning: you have specified the global option -maxdepth after the argument -iname, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/kaggle/input/datasets/fedesoriano/cifar100


In [14]:
import os, shutil

src = "/kaggle/input/datasets/fedesoriano/cifar100"
dst = os.path.expanduser("~/.cache/datasets/cifar-100-python")
os.makedirs(dst, exist_ok=True)

for fname in ["train", "test", "meta"]:
    shutil.copy(os.path.join(src, fname), os.path.join(dst, fname))

print(os.listdir(dst))

['test', 'train', 'meta']


In [15]:
import torchvision
ds = torchvision.datasets.CIFAR100(root=os.path.expanduser("~/.cache/datasets"), train=True, download=False)
print(len(ds))

50000


In [ ]:
%cd /kaggle/working/hgvt
!PRECISION="bf16" ATTN_PRECISION="fp32" USE_XFORMERS=1 USE_APEX=0 \
python train.py --precision bf16 --gpus 1 \
    --name smoke_test \
    --logdir /kaggle/working/logs \
    --config configs/hgvt_mu.yaml \
    --max_steps 20 \
    --limit_train_batches 4 \
    --limit_val_batches 2

In [16]:
!find /kaggle/input -iname "*imagenet100*" -maxdepth 3

find: warning: you have specified the global option -maxdepth after the argument -iname, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/kaggle/input/datasets/ambityga/imagenet100


In [17]:
!find /kaggle/input/datasets/ambityga/imagenet100 -maxdepth 2

/kaggle/input/datasets/ambityga/imagenet100
/kaggle/input/datasets/ambityga/imagenet100/Labels.json
/kaggle/input/datasets/ambityga/imagenet100/train.X1
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01531178
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01440764
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01494475
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01950731
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01795545
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01632777
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n02012849
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01775062
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n02007558
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01484850
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01930112
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n01984695
/kaggle/input/datasets/ambityga/imagenet100/train.X1/n02037110
/kaggle/input/datasets/ambit

In [18]:
%%writefile /kaggle/working/hgvt/hgvt/data/torchvision_class_datasets.py
import torchvision
import os
from abc import ABC, abstractmethod
from torch.utils.data import Dataset, Subset
import random

# Registry to store dataset creation functions
DATASET_REGISTRY = {}

def register_dataset(name):
    """Decorator to register dataset in the registry"""
    def decorator(cls):
        DATASET_REGISTRY[name] = cls
        return cls
    return decorator
    

class BaseDataset(ABC):
    """Abstract base class for datasets, should be subclassed for each dataset."""
    
    def __init__(self, root="~/.cache/datasets/"):
        self.seed = None
        self.root = root
    
    def setup(self, seed=None):
        """Set up the dataset, including applying the seed if provided."""
        self.seed = seed if seed is not None else 21 # Save the seed for deterministic splitting
        pass

    @abstractmethod
    def create_train(self, batch_size=None):
        """Create and return the training dataset"""
        pass

    @abstractmethod
    def create_val(self, batch_size=None):
        """Create and return the validation dataset"""
        pass

    @abstractmethod
    def create_test(self, batch_size=None):
        """Create and return the test dataset"""
        pass

    def _deterministic_shuffle(self, dataset, split_size):
        """Create a deterministic shuffle of the dataset indices and return a subset."""
        if self.seed is not None:
            rng = random.Random(self.seed)
        else:
            rng = random
        indices = list(range(len(dataset)))
        rng.shuffle(indices)
        subset_indices = indices[:split_size]
        return Subset(dataset, subset_indices)

@register_dataset("CIFAR10")
class CIFAR10Dataset(BaseDataset):
    def setup(self, seed=None):
        """Download CIFAR10 dataset and set seed."""
        super().setup(seed)
        torchvision.datasets.CIFAR10(root=self.root, train=True, download=True)
        torchvision.datasets.CIFAR10(root=self.root, train=False, download=True)

    def create_train(self, batch_size=None):
        """Return the training dataset for CIFAR10 (PIL images)."""
        return torchvision.datasets.CIFAR10(root=self.root, train=True, transform=None)

    def create_val(self, batch_size=None):
        """Return a deterministic validation dataset for CIFAR10."""
        test_full = torchvision.datasets.CIFAR10(root=self.root, train=False, transform=None)
        if batch_size is None:
            return test_full
        # otherwise pull off two batch splits from test for val
        val_size = min(2 * batch_size, len(test_full))
        return self._deterministic_shuffle(test_full, val_size)

    def create_test(self, batch_size=None):
        """Return the test dataset for CIFAR10 (PIL images)."""
        return torchvision.datasets.CIFAR10(root=self.root, train=False, transform=None)


@register_dataset("CIFAR100")
class CIFAR100Dataset(BaseDataset):
    def setup(self, seed=None):
        """Download CIFAR100 dataset and set seed."""
        super().setup(seed)
        torchvision.datasets.CIFAR100(root=self.root, train=True, download=True)
        torchvision.datasets.CIFAR100(root=self.root, train=False, download=True)

    def create_train(self, batch_size=None):
        """Return the training dataset for CIFAR100 (PIL images)."""
        return torchvision.datasets.CIFAR100(root=self.root, train=True, transform=None)

    def create_val(self, batch_size=None):
        """Return a deterministic validation dataset for CIFAR100."""
        test_full = torchvision.datasets.CIFAR100(root=self.root, train=False, transform=None)
        if batch_size is None:
            return test_full
        # otherwise pull off two batch splits from test for val
        val_size = min(2 * batch_size, len(test_full))
        return self._deterministic_shuffle(test_full, val_size)

    def create_test(self, batch_size=None):
        """Return the test dataset for CIFAR100 (PIL images)."""
        return torchvision.datasets.CIFAR100(root=self.root, train=False, transform=None)

    def identity_transform(self):
        """Return an identity transform to ensure that PIL images are passed from the dataset."""
        return torchvision.transforms.Lambda(lambda x: x)  # Return image without modification


class MergedImageFolderDataset(Dataset):
    """
    Merges one or more ImageFolder-style root directories (each containing
    a subset of class subfolders) into a single dataset using one
    consistent class_to_idx mapping supplied by the caller.

    This exists because the ambityga/imagenet100 Kaggle dataset splits its
    100 training classes across 4 separate folders (train.X1..train.X4),
    each holding only ~25 class subfolders. A plain torchvision.ImageFolder
    on any single one of those folders would only see its own subset of
    classes and assign inconsistent integer labels across folders.
    """
    def __init__(self, roots, class_to_idx, transform=None,
                 loader=torchvision.datasets.folder.default_loader,
                 extensions=torchvision.datasets.folder.IMG_EXTENSIONS):
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.loader = loader
        self.samples = []
        for root in roots:
            for class_name in sorted(os.listdir(root)):
                class_dir = os.path.join(root, class_name)
                if not os.path.isdir(class_dir) or class_name not in class_to_idx:
                    continue
                label = class_to_idx[class_name]
                for fname in sorted(os.listdir(class_dir)):
                    if fname.lower().endswith(extensions):
                        self.samples.append((os.path.join(class_dir, fname), label))
        if len(self.samples) == 0:
            raise RuntimeError(f"No images found under roots: {roots}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = self.loader(path)
        if self.transform is not None:
            image = self.transform(image)
        return image, label


@register_dataset("ImageNet100")
class ImageNet100Dataset(BaseDataset):
    """
    Expects the ambityga/imagenet100 Kaggle dataset layout:
        root/train.X1/<wordnet_code>/*.JPEG
        root/train.X2/<wordnet_code>/*.JPEG
        root/train.X3/<wordnet_code>/*.JPEG
        root/train.X4/<wordnet_code>/*.JPEG
        root/val.X/<wordnet_code>/*.JPEG

    Since Kaggle attaches this dataset read-only, `root` defaults directly
    to the Kaggle input mount path rather than the usual ~/.cache/datasets/
    (there is nothing to download/copy). Override by constructing this
    class with a different `root` if your mount path differs.
    """
    def __init__(self, root="/kaggle/input/datasets/ambityga/imagenet100"):
        super().__init__(root=root)
        self.classes = None
        self.class_to_idx = None
        self.train_dirs = None
        self.val_dir = None

    def setup(self, seed=None):
        super().setup(seed)
        train_dirs = [os.path.join(self.root, f"train.X{i}") for i in range(1, 5)]
        for d in train_dirs:
            if not os.path.isdir(d):
                raise FileNotFoundError(f"Expected ImageNet100 train dir not found: {d}")
        val_dir = os.path.join(self.root, "val.X")
        if not os.path.isdir(val_dir):
            raise FileNotFoundError(f"Expected ImageNet100 val dir not found: {val_dir}")

        # Build one global class_to_idx from the sorted union of class
        # folders across all train.X* dirs, so labels are consistent
        # regardless of which train.X folder a given class lives in.
        all_classes = set()
        for d in train_dirs:
            all_classes.update(
                name for name in os.listdir(d) if os.path.isdir(os.path.join(d, name))
            )
        self.classes = sorted(all_classes)
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.train_dirs = train_dirs
        self.val_dir = val_dir

    def _ensure_setup(self):
        if self.class_to_idx is None:
            self.setup(seed=self.seed)

    def create_train(self, batch_size=None):
        self._ensure_setup()
        return MergedImageFolderDataset(self.train_dirs, self.class_to_idx, transform=None)

    def create_val(self, batch_size=None):
        self._ensure_setup()
        # val.X already contains all 100 classes in one folder, but reuse
        # MergedImageFolderDataset for a consistent (path, label) interface
        return MergedImageFolderDataset([self.val_dir], self.class_to_idx, transform=None)

    def create_test(self, batch_size=None):
        # no separate held-out test split is provided by this dataset;
        # reuse the validation split
        return self.create_val(batch_size)

Overwriting /kaggle/working/hgvt/hgvt/data/torchvision_class_datasets.py


In [19]:
%%writefile /kaggle/working/hgvt/configs/hgvt_lt_in100_reduced_data.yaml
model:
  base_learning_rate: 1.0 # LR scheduler multiplier
  target: hgvt.hypergraph.HyperGraph
  params:
    monitor: "val/ce_loss"
    save_top_k: 1
    use_ema: False
    ema_decay: 0.99996
    log_finegrained: False
    compile_model: False
    log_experts: False
    # === Criteron args ===
    label_smoothing: 0.1   # cross-entropy label smoothing
    mixup_active: True     # mixup is active when training, so we need to change the CE method
    # === loss weights ===
    diversity_weight: 1.0
    pop_weight: 1.0
    exp_weight: 1.0
    # === optimizer parameters ===
    opt_beta1: 0.9
    opt_beta2: 0.999
    opt_weight_decay: 0.05
    opt_eps: 1.0e-8
    # Note: for ImageNet-100, each epoch is ~244 steps @ batch=512

    timm_optimizer: # opt for timm
        opt: adamw    # switched from fusedadamw (requires apex, not installed here)
        lr: 1e-3
        opt_eps: 1e-8
        weight_decay: .05
        opt_betas: [0.9, 0.999]
        momentum: 0.9

    # note have to divide numbers by 2 for 2 repeats (repeat_aug=2 below)
    scheduler_config: # for timm
        sched: cosine
        epochs: 100 # corresponds to ~200 real epochs due to repeat_aug=2
        warmup_epochs: 8
        cooldown_epochs: 6
        warmup_lr: 1e-6
        min_lr: 1e-5
        lr_noise: Null
        lr_noise_pct: 0.67
        lr_noise_std: 1.0
        seed: 21
        lr_cycle_mul: 1
        lr_cycle_limit: 1

    model_config:
        target:  hgvt.network.hypergraph.HypergraphNet
        params:
            # === I/O params ===
            in_features: 3
            image_size: 160
            patch_size: 16   # num_nodes = 100
            num_classes: 100
            # === Arch params ===
            num_layers: 12
            repeat_n: Null
            hidden_dim: 128
            num_heads: 4 # 128/4 = 32
            adj_dim: 64
            hnode_capacity: 12 # hidden nodes to 16
            edge_capacity: 32 # edge capacity to 50
            hedge_capacity: 6 # hidden edges to 8
            # === primary state config ===
            split_mode: True
            shared_adjproj: True
            joint_mlp: True
            bind_qkv: False
            use_stem: True
            use_nodesa: True
            correct_imgcond: True
            div_hdim: 1
            # === regularization helpers ===
            dropout: 0.0
            drop_path: 0.1
            drop_decay: True
            skip_last: True
            weight_init: True
            virtual_init: False
            train_pe: False
            use_virtual_pemb: True
            use_diversity_loss: True
            output_ln: True
            use_xformers: False
            correct_mod: True
            pop_level: [20.7, 0.5, 0.5]
            # == pooling params ===
            pool_type: 'bothEd'
            expert_topk: 1
            expert_ce_weight: 0.1
            expert_noise_level: 1e-1
            expert_dropout: 0.1
            expert_label_smoothing: 0.0
            expert_use_gate: False

data:
  target: hgvt.data.prefetch_loader.PrefetchLoaderFromConfig
  params:
    use_async: False   # matches the proven-working CIFAR-100 TorchDataModule setup
    batch_size: 128    # reduced from 512 to fit on a ~14.5GB GPU at 160x160; see accumulate_grad_batches below
    async_config:
        train:
            fp16: True
            # === random erase ===
            re_mode: 'pixel'
            re_prob: 0.25
            re_count: 1
            re_num_splits: 0
            # === mixup args ===
            label_smoothing: 0.1
            mixup: 0.8
            cutmix: 1.0
            cutmix_minmax: Null
            mixup_prob: 0.8
            mixup_switch_prob: 0.5
            mixup_mode: 'batch'
            mixup_off_epoch: 0
            num_classes: 100
        val:
            fp16: True
        test:
            fp16: True

    factory_config:
        target: hgvt.data.torchvision_data_module.TorchDataModule
        params:
            batch_size: 128
            dataset_name: ImageNet100
            num_workers: 4
            multinode: False
            pin_memory: True
            generate_cls: True
            train:
              shuffle: 10000
              repeat: 1
              repeat_aug: 2
              transforms:
                  img_size: 160
                  color_jitter: 0.4
                  re_mode: 'pixel'
                  re_prob: 0.25
                  re_count: 1
                  re_split: False
                  scale: [0.08, 1.0]
                  ratio: [0.75, 1.33]
                  hflip: 0.5
                  vflip: 0.0
                  num_aug_splits: 0
                  auto_augment: 'rand-m9-mstd0.5-inc1'
                  interpolation: 'random'
                  use_prefetcher: True
                  crop_pct: Null

            validation:
              shuffle: 0
              repeats: 1
              transforms:
                  img_size: 160
                  no_aug: True
                  use_prefetcher: True

            test:
              shuffle: 0
              repeats: 1
              transforms:
                  img_size: 160
                  no_aug: True
                  use_prefetcher: True

lightning:
  modelcheckpoint:
    params:
      # ~4 checkpoints across the corrected total step count below
      every_n_train_steps: 1250

  trainer:
    accelerator: gpu
    num_sanity_val_steps: 0
    # REDUCED-DATASET RUN: run this config with --limit_train_batches 0.05
    # (recomputed below; NOT 0.1 as originally instructed) at the CLI.
    #
    # CORRECTED MATH: repeat_aug=2 in factory_config.params.train halves
    # the DataLoader's effective micro-batch size (128/2=64), so a full
    # epoch over 130,000 images is ceil(130000/64) = 2032 micro-batches,
    # not ~1016 as originally (wrongly) assumed. At limit_train_batches
    # 0.05: 0.05 x 2032 = ~101 micro-batches/epoch, / accumulate_grad_batches(4)
    # = ~25 optimizer steps/epoch, x 200 real epochs = ~5050 steps total.
    #
    # The scheduler_config above is UNCHANGED from the full run (100
    # scheduler-epochs = 200 real epochs, warmup=8, cooldown=6) because
    # timm's create_scheduler (v1 API used here) operates purely on epoch
    # *count*, not steps-per-epoch — so shrinking the dataset via
    # limit_train_batches keeps the exact same warmup/cosine/cooldown
    # shape, just compressed in wall-clock time.
    check_val_every_n_epoch: 10
    benchmark: True
    # 101 micro-batches/epoch x 200 epochs / 4 accumulation = ~5050 steps.
    # If you change --limit_train_batches, recompute:
    #   micro_batches_per_epoch = int(limit_train_batches * 2032)
    #   max_steps = (micro_batches_per_epoch * 200) // accumulate_grad_batches
    max_steps: 5050
    max_epochs: 200   # cosmetic only (fixes "Epoch N/999" display); max_steps above is what actually stops training
    accumulate_grad_batches: 4   # 128 (physical) x 4 = 512 effective batch size
    gradient_clip_val: 1.0

Writing /kaggle/working/hgvt/configs/hgvt_lt_in100_reduced_data.yaml


In [20]:
import os
root = "/kaggle/input/datasets/ambityga/imagenet100"
total = 0
for i in range(1, 5):
    d = os.path.join(root, f"train.X{i}")
    for cls in os.listdir(d):
        total += len(os.listdir(os.path.join(d, cls)))
print("Total train images:", total)
print("~10% per epoch:", int(total * 0.1))

Total train images: 130000
~10% per epoch: 13000


In [21]:
path = "/kaggle/working/hgvt/hgvt/data/prefetch_loader.py"
with open(path) as f:
    content = f.read()

old = """    # don't include the len property since wds doesn't have it
    #def __len__(self):
    #    return len(self.loader)"""

new = """    # Proxied to the wrapped loader. For map-style datasets (e.g. via
    # TorchDataModule) self.loader has a real __len__, which PyTorch
    # Lightning needs to interpret `limit_train_batches` as a fraction
    # (e.g. 0.1) rather than requiring an int or 1.0. For WebDataset-based
    # IterableDataset loaders that have no natural length, len(self.loader)
    # will simply raise TypeError here, same as if __len__ didn't exist.
    def __len__(self):
        return len(self.loader)"""

assert old in content, "pattern not found — check whitespace, or the earlier batch_sampler patch already changed this block"
content = content.replace(old, new)
with open(path, "w") as f:
    f.write(content)
print("patched")

patched


In [22]:
import os
root = "/kaggle/input/datasets/ambityga/imagenet100"
total = 0
for i in range(1, 5):
    d = os.path.join(root, f"train.X{i}")
    for cls in os.listdir(d):
        total += len(os.listdir(os.path.join(d, cls)))
print("Total train images:", total)

Total train images: 130000


In [ ]:
%cd /kaggle/working/hgvt
!PRECISION="bf16" ATTN_PRECISION="fp32" USE_XFORMERS=1 USE_APEX=0 \
python train.py --precision bf16 --gpus 1 \
    --name in100_reduced \
    --logdir /kaggle/working/logs \
    --config configs/hgvt_lt_in100_reduced_data.yaml \
    --limit_train_batches 0.05

In [23]:
%%writefile /kaggle/working/hgvt/eval.py
"""
eval.py — standalone evaluation entry point for HgVT checkpoints.

Like train.py, this is NOT part of the original RGenDiff/hgvt release —
the repo ships no evaluation script. This one loads a trained checkpoint
and reports Top-1/Top-5 accuracy on the FULL validation set (ignoring any
--limit_val_batches used during training, since that was only for
periodic monitoring, not a final reported metric).

Usage:
    PRECISION="bf16" && \\
    export ATTN_PRECISION="fp32" && \\
    export USE_XFORMERS=1 && \\
    export USE_APEX=0 && \\
    python eval.py \\
        --config configs/hgvt_lt_in100_reduced_data.yaml \\
        --checkpoint /path/to/epoch144-step3750.ckpt \\
        --gpus 1 \\
        --precision bf16

Note: --config must be the SAME config (same model_config) used to
produce the checkpoint, since the checkpoint file only contains weights,
not architecture. The data config's augmentation settings for train are
irrelevant here; only the validation transforms are used.
"""

import argparse
import os

from omegaconf import OmegaConf
import torch
import pytorch_lightning as pl

from hgvt.util import instantiate_from_config


def get_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--config", type=str, required=True,
        help="Path to the config yaml used to train the checkpoint (defines model architecture + data).",
    )
    parser.add_argument(
        "--checkpoint", type=str, required=True,
        help="Path to a .ckpt file to evaluate.",
    )
    parser.add_argument(
        "--gpus", type=int, default=1,
        help="Number of GPUs to use. 0 = CPU.",
    )
    parser.add_argument(
        "--precision", type=str, default="32",
        help="Evaluation precision: 32, 16, or bf16.",
    )
    return parser


def normalize_precision(precision_arg):
    mapping = {"32": "32", "16": "16", "bf16": "bf16"}
    return mapping.get(precision_arg, precision_arg)


def build_trainer(args):
    use_gpu = args.gpus > 0 and torch.cuda.is_available()
    if args.gpus > 0 and not torch.cuda.is_available():
        print("WARNING: --gpus > 0 requested but no CUDA device is visible. Falling back to CPU.")

    accelerator = "gpu" if use_gpu else "cpu"
    devices = args.gpus if use_gpu else 1
    precision = normalize_precision(args.precision)

    try:
        trainer = pl.Trainer(
            accelerator=accelerator,
            devices=devices,
            precision=precision,
            logger=False,
        )
    except Exception as e:
        if precision in ("16", "bf16"):
            print(f"Trainer construction failed with precision='{precision}' ({e}); "
                  f"retrying with '{precision}-mixed'.")
            trainer = pl.Trainer(
                accelerator=accelerator,
                devices=devices,
                precision=f"{precision}-mixed",
                logger=False,
            )
        else:
            raise
    return trainer


def main():
    parser = get_parser()
    args = parser.parse_args()

    if not os.path.isfile(args.checkpoint):
        raise FileNotFoundError(f"Checkpoint not found: {args.checkpoint}")

    config = OmegaConf.load(args.config)
    model_config = config.model
    data_config = config.data

    # --- build model (architecture only; weights loaded via ckpt_path below) ---
    model = instantiate_from_config(model_config)

    # --- build data module ---
    datamodule = instantiate_from_config(data_config)

    inner_factory = getattr(datamodule, "dataset_factory", None)
    if inner_factory is not None and hasattr(inner_factory, "prepare_data"):
        print("Preparing dataset via inner data factory...")
        inner_factory.prepare_data()

    # --- trainer (validation only, full val set, no limit_val_batches) ---
    trainer = build_trainer(args)

    print(f"Evaluating checkpoint: {args.checkpoint}")
    results = trainer.validate(model=model, datamodule=datamodule, ckpt_path=args.checkpoint)

    print("\n=== Evaluation results ===")
    for r in results:
        for k, v in r.items():
            print(f"{k}: {v}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/hgvt/eval.py


In [ ]:
!find /kaggle/input -iname "*.ckpt"

In [24]:
%cd /kaggle/working/hgvt
!PRECISION="bf16" ATTN_PRECISION="fp32" USE_XFORMERS=1 USE_APEX=0 \
python eval.py \
    --config configs/hgvt_lt_in100_reduced_data.yaml \
    --checkpoint /kaggle/input/datasets/sharifnbd/checkpointalgo/epoch144-step3750.ckpt \
    --gpus 1 \
    --precision bf16

/kaggle/working/hgvt
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
no module 'xformers'. Processing without...
 Env Configs:
	 _ATTN_PRECISION==fp32 True
	 _USE_APEX==False
	 _USE_XFORMERS==False
	 _NORMALIZE_QK==True
---------------------------------------
Constructing hypergraph with
 - 100 nodes
 - 12 hidden nodes
 - 32 edges
 - 6 hidden edges
Using ViGStem patcher for patch_size=16 with levels=3
Building transformer with 12 layers, d=128, d_a=64, and split_mode=True
setting stochastic drop_path to: [0.0, 0.00909090880304575, 0.0181818176060915, 0.027272727340459824, 0.036363635212183, 0.045454543083906174, 0.054545458406209946, 0.06363636255264282, 0.0727272778749466, 0.08181818574666977, 0.09090909361839294, 0.10000000149011612]
Making HyperGraphBlock, ski